In [ ]:
import boto3
from pyspark.sql.types import (
    StructType,
)
import json
from datetime import datetime, timedelta
import os
from typing import List, Dict, Any

In [ ]:
# Initialize CloudWatch client
AWS_REGION = os.getenv("AWS_REGION", "eu-west-2")
session = boto3.Session(profile_name="nhsd-nrlf-dev", region_name=AWS_REGION)
client = session.client("logs")

In [ ]:
def parse_json(log):
    try:
        return json.loads(log)
    except json.JSONDecodeError:
        return {"raw_message": str(log)}


def fetch_cloudwatch_logs(log_group_name, start_time, end_time):
    logs = []
    next_token = None
    while True:
        if next_token:
            response = client.filter_log_events(
                logGroupName=log_group_name,
                startTime=int(start_time.timestamp() * 1000),
                endTime=int(end_time.timestamp() * 1000),
                nextToken=next_token,
            )
        else:
            response = client.filter_log_events(
                logGroupName=log_group_name,
                startTime=int(start_time.timestamp() * 1000),
                endTime=int(end_time.timestamp() * 1000),
            )
        for event in response["events"]:
            logs.append(
                parse_json(event["message"])
            )
        next_token = response.get("nextToken")
        if not next_token:
            break
    # print(logs)
    return logs

In [ ]:
# Example usage
log_group = "searchPostDocumentReference"
log_group_name = f"/aws/lambda/nhsd-nrlf--dev-1--api--consumer--{log_group}"
end_time = datetime.now() - timedelta(days=25)
start_time = end_time - timedelta(days=21)  # Fetch logs from the last hour
logs = fetch_cloudwatch_logs(log_group_name, start_time, end_time)
print(len(logs))

In [ ]:
print(logs)

In [ ]:
def checktype(obj: List):
    # print(obj)
    if obj:
        data_types = [type(i).__name__ for i in obj]
        return max(set(data_types), key=data_types.count)
    else:
        return "str"


def _is_struct(field_type: Any) -> bool:
    """Check if a field type is a struct"""
    return isinstance(field_type, dict) and field_type.get("type") == "struct"


def find_cols(log, prefix=""):
    cols = {}
    if isinstance(log, dict):
        for k, v in log.items():
            # print(f"k: {k}, v: {v}")
            name = f"{prefix}__{k}" if prefix else k
            if isinstance(v, dict):
                cols.update(find_cols(v, name))

            if name not in cols.keys() and v:
                if isinstance(v, list):
                    cols[name] = f"list.{checktype(v)}"
                    temp = {}
                    for i in v:
                        if checktype(v) == "dict":
                            for key, val in i.items():
                                temp_name = f"{name}__{key}"
                                if isinstance(val, dict):
                                    temp.update(find_cols(val, temp_name))
                                if isinstance(val, list):
                                    temp.update(find_cols(val, temp_name))
                                if temp_name not in temp and val:
                                    temp[temp_name] = type(val).__name__
                    cols.update(temp)

                else:
                    cols[name] = type(v).__name__

    elif isinstance(log, list):
        if prefix not in cols:
            cols[prefix] = f"list.{checktype(log)}"
        list_temp = {}
        for item in log:
            if checktype(log) == "dict":
                for list_key, list_val in item.items():
                    list_temp_name = f"{prefix}__{list_key}"
                    if isinstance(list_val, dict):
                        list_temp.update(find_cols(list_val, list_temp_name))
                    if isinstance(list_val, list):
                        list_temp.update(find_cols(list_val, list_temp_name))
                    if list_temp_name not in list_temp and list_val:
                        list_temp[list_temp_name] = type(list_val).__name__
        cols.update(list_temp)

    return cols


fields = {}
for log in logs:
    fields.update(find_cols(log))

print(fields)


def format_schema(fields: Dict):
    final = {}
    type_mapping = {
        "str": "string",
        "bool": "boolean",
        "NoneType": None,
        "int": "long",
        "float": "double",
        "dict": {"fields": {}, "type": "struct"},
        "list.str": {"type": "array", "elementType": "string", "containsNull": True},
        "list.bool": {"type": "array", "elementType": "boolean", "containsNull": True},
        "list.int": {"type": "array", "elementType": "long", "containsNull": True},
        "list.NoneType": {"type": "array", "elementType": None, "containsNull": True},
        "list.float": {"type": "array", "elementType": "double", "containsNull": True},
        "list.dict": {
            "type": "array",
            "elementType": {"fields": {}, "type": "struct"},
            "containsNull": True,
        },
    }

    for k, v in fields.items():
        if v not in type_mapping:
            raise ValueError(
                f"Unsupported type '{v}' for field '{k}'. "
                f"Allowed types : {type_mapping.keys()}"
            )
        if "__" not in k:  # if base level, then add accordingly
            if k in final.keys():
                if not _is_struct(final[k]["type"]):
                    raise ValueError(
                        f"Field {k} already exists as non-struct type "
                        f"({final[k]['type']}). Cannot redefine"
                    )
            else:
                final[k] = {
                    "metadata": {},
                    "name": k,
                    "nullable": True,
                    "type": type_mapping[v],
                }
        else:  # else not base level, so form hierarchy
            levels = k.split("__")
            end = len(levels) - 1
            hierarchy = {}
            for i, level in enumerate(levels):  # iterate through levels
                if i < end:
                    data_type = {"fields": {}, "type": "struct"}
                else:
                    if v == "list.dict":
                        data_type = {
                            "type": "array",
                            "elementType": {"fields": {}, "type": "struct"},
                            "containsNull": True,
                        }
                    else:
                        data_type = type_mapping[v]
                if i == 0:
                    if level not in final.keys():
                        final[level] = {
                            "metadata": {},
                            "name": level,
                            "nullable": True,
                            "type": data_type,
                        }

                    hierarchy = final[level]["type"]["fields"]
                else:
                    try:
                        if levels[i - 1] in hierarchy:
                            if "elementType" not in hierarchy[levels[i - 1]]["type"]:
                                hierarchy = hierarchy[levels[i - 1]]["type"]["fields"]
                            else:
                                hierarchy = hierarchy[levels[i - 1]]["type"][
                                    "elementType"
                                ]["fields"]
                    except:
                        raise ValueError(f"{hierarchy[levels[i-1]]}")
                    if level not in hierarchy:
                        hierarchy[level] = {
                            "metadata": {},
                            "name": level,
                            "nullable": True,
                            "type": data_type,
                        }
                data_type = None

    return final


l = format_schema(fields)
print(l)

In [ ]:
def make_fields(final):
    for k, v in final.items():
        # print(f"k: {k}, v: {v}")
        parent = k
        if isinstance(v["type"], dict):
            if "elementType" in v["type"]:
                if isinstance(v["type"]["elementType"], dict):
                    fields = make_fields(v["type"]["elementType"]["fields"])
                    final[parent]["type"]["elementType"]["fields"] = fields

            elif not isinstance(v["type"]["fields"], list):
                fields = make_fields(v["type"]["fields"])
                final[parent]["type"]["fields"] = fields

    return list(final.values())


jsonSchema = {"fields": make_fields(l), "type": "struct"}
# jsonSchema

In [ ]:
schema = StructType.fromJson(jsonSchema)
schema